In [3]:
from google.oauth2 import service_account
#from googleapiclient.discovery import build
from sodapy import Socrata
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import os
import numpy as np
import pandas as pd
# from rdflib import Graph, Literal, Namespace, URIRef
# from rdflib.namespace import RDF, RDFS, DCTERMS,SKOS

from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import DCTERMS, FOAF, RDF, XSD


bic_etl_home = os.environ.get("bic_etl_home")

In [4]:
def getDatasetTrackerInfo(bic_etl_home):

    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
                 "https://www.googleapis.com/auth/drive.file",
                      "https://www.googleapis.com/auth/drive"]
    
    #creds = ServiceAccountCredentials.from_json_keyfile_name('../../scripts/client_secret.json',
    #    scope)
    creds = ServiceAccountCredentials.from_json_keyfile_name(os.path.join(bic_etl_home, 'general', 'scripts','client_secret.json'),scope)
    
    client = gspread.authorize(creds)
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
    dfTracker = pd.DataFrame(tracker.get_all_records(head=2))
    return dfTracker

def getCimDatasets():
    cim_url_query = "data.colorado.gov"
    allDatasets = []
    cimDatasets = {}
    
    # Connect to the Socrata API
    with Socrata(cim_url_query, None) as client:
        datasets = client.datasets()
        for dataset in datasets:
            allDatasets.append(dataset)
            if dataset['owner']['display_name'] == 'Colorado Information Marketplace' or dataset['owner']['display_name'] == "Business Intelligence Center of CO":
                title=dataset["resource"]["name"]
                w4x4=dataset["resource"]["id"]

                cimDatasets[w4x4]=dataset
    
    return cimDatasets

def link_exists(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False



In [5]:
track=getDatasetTrackerInfo(bic_etl_home)

In [8]:
cimDatasets = getCimDatasets()

In [9]:
track

,Dataset Title,Short Description,Category,Keywords,Type,License Type,Data Provider,Data Provided by,Source Link,State Steward,...,Data Owner - Name,Data Owner - Position,Data Owner - Phone Number,Data Owner - Email,Data Steward - Name,Data Steward - Position,Data Steward - Phone Number,Data Steward - Email,Name of Person Providing Data,Email of Person Providing Data
0,Road Traffic Counts in Colorado 2019,Traffic counts on public roads under local jur...,Transportation,"bic, gocodecolorado, colorado, cdot, colorado ...",Traffic,Public Domain,CDOT,CDOT - Colorado Department of Transportation,http://dtdapps.coloradodot.info/otis/trafficdata,CDOT,...,Gary Aucott,GIS Support Unit Manager,303.512.4447,gary.aucott@state.co.us,Nathan Rogers,,,nathanial.rogers@state.co.us,Nathan Rogers,nathanial.rogers@state.co.us
1,Liquor Compliance Check Statistics in Colorado,"Business names, location, liquor license type ...",Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,Michelle Brown,Director Liquor Enforcement Division,303-205-2306,michellen.brown@state.co.us,Evelyne Nottage,Program Assisstnat,303-205-2310,evelyne.nottage@state.co.us,Janina Rivera/ New Evelyne Nottage,evelyne.nottage@state.co.us
2,Recently Expired and Surrendered Liquor Licens...,"Names, locations, license number and status of...",Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,Michelle Brown,Director Liquor Enforcement Division,303-205-2306,michellen.brown@state.co.us,Evelyne Nottage,Program Assistant,303-205-2310,evelyne.nottage@state.co.us,Evelyne Nottage,evelyne.nottage@state.co.us
3,Liquor Licenses in Colorado,Names and locations of business with active li...,Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,Michelle Brown,Director Liquor Enforcement Division,303-205-2306,michellen.brown@state.co.us,Evelyne Nottage,Program Assistant,303-205-2310,evelyne.nottage@state.co.us,Janina Rivera/ New Evelyne Nottage,evelyne.nottage@state.co.us
4,Recently Approved Liquor Licenses in Colorado,Names and locations of business with active li...,Regulations,"bic, gocodecolorado, colorado, dor, department...",Liquor Regulation,Public Domain,CDOR,CDOR - Colorado Department of Revenue,https://www.colorado.gov/pacific/enforcement/l...,CDOR,...,Michelle Brown,Director Liquor Enforcement Division,303-205-2306,michellen.brown@state.co.us,Evelyne Nottage,Program Assistant,303-205-2310,evelyne.nottage@state.co.us,Evelyne Nottage,evelyne.nottage@state.co.us
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,Colorado State Agency Natural Gas Use FY15 - FY21,Colorado State agency natural gas use measured...,State,"gocode, gocodecolorado, bic, energy, fossil fuels",,Public Domain,GOV,GOV - Colorado Energy Office,https://energyoffice.colorado.gov/,GOV,...,,,,,,,,,,
469,Colorado State Agency Renewable Energy Use FY1...,"Colorado State agency renewable energy use, me...",State,"gocode, gocodecolorado, bic, electricity, rene...",,Public Domain,GOV,GOV - Colorado Energy Office,https://energyoffice.colorado.gov/,GOV,...,,,,,,,,,,
470,Managed Access Lanes in Colorado,"Managed Access Lanes, HOV, Tolls, Parking. Dat...",Government,"cdot, colorado department of transportation",Infrastructure,Public Domain,CDOT,CDOT - Colorado Department of Energy,https://www.codot.gov/,,...,Phyllis Sneider,,,phyllis.snider@state.co.us,,,,,,
471,Tree Canopy Assesment 2013 Denver,Metro Denver City boundary polygons with attri...,Natural Resources,"bic, tree, trees, canopy, tree canopy, denver,...",Natural Features,Public Domain,City and county of D

In [10]:
def create_catalog_graph(
    catalog_uri: str,
    title: str,
    description: str,
    homepage: str,
    publisher_uri: str,
    publisher_name: str,
    base_graph: Graph | None = None,
) -> Graph:
    """
    Create/augment a graph with a DCAT Catalog and link existing dcat:Dataset nodes to it.
    If base_graph is provided, it will be used/augmented; otherwise a new Graph is created.
    """
    dcat_graph = Graph()
    DCAT = Namespace("http://www.w3.org/ns/dcat#")
    FOAF = Namespace("http://xmlns.com/foaf/0.1/")

    g = base_graph if base_graph is not None else Graph()

    # Bind prefixes for readability
    g.bind("dcat", DCAT)
    g.bind("dct", DCTERMS)
    g.bind("foaf", FOAF)

    catalog = URIRef(catalog_uri)
    publisher = URIRef(publisher_uri)

    # Publisher agent (FOAF Organization)
    g.add((publisher, RDF.type, FOAF.Organization))
    g.add((publisher, FOAF.name, Literal(publisher_name)))

    # Catalog node + core metadata
    g.add((catalog, RDF.type, DCAT.Catalog))
    g.add((catalog, DCTERMS.title, Literal(title, lang="en")))
    g.add((catalog, DCTERMS.description, Literal(description, lang="en")))
    g.add((catalog, DCTERMS.publisher, publisher))
    g.add((catalog, FOAF.homepage, URIRef(homepage)))
    # Optional dating (set to "today" or your real dates)
    # g.add((catalog, DCTERMS.issued, Literal("2025-09-09", datatype=XSD.date)))
    # g.add((catalog, DCTERMS.modified, Literal("2025-09-09", datatype=XSD.date)))

    # Link all existing datasets in the graph to the catalog
    for s in g.subjects(RDF.type, DCAT.Dataset):
        g.add((catalog, DCAT.dataset, s))

    return DCAT,g

def addDcatDataset(DCAT, dcat_graph, title, description, url, download_url):
    dataset_uri = URIRef(url)

    # Add basic dataset information
    dcat_graph.add((dataset_uri, RDF.type, DCAT.Dataset))
    dcat_graph.add((dataset_uri, DCTERMS.title, Literal(title, lang="en")))
    dcat_graph.add((dataset_uri, DCTERMS.description, Literal(description, lang="en")))

    # Add language information
    english_lang_uri = URIRef("http://publications.europa.eu/resource/authority/language/ENG")
    dcat_graph.add((dataset_uri, DCTERMS.language, english_lang_uri))

    # Create a unique distribution for this dataset
    if link_exists(download_url):
        distribution_uri = URIRef(download_url)
        dcat_graph.add((dataset_uri, DCAT.distribution, distribution_uri))
        dcat_graph.add((distribution_uri, RDF.type, DCAT.Distribution))
        dcat_graph.add((distribution_uri, DCTERMS.title, Literal("Download CSV", lang="en")))
        dcat_graph.add((distribution_uri, DCTERMS.language, english_lang_uri))
        dcat_graph.add((distribution_uri, DCAT.mediaType, Literal("text/csv")))
        dcat_graph.add((distribution_uri, DCAT.downloadURL, distribution_uri))
        dcat_graph.add((distribution_uri, DCTERMS.description, Literal(
            "API download link (default 1000 row limit; use $limit=N to override). See Socrata API documentation for details.",
            lang="en")))
    else:
        print(f"Warning: Download URL {download_url} is not accessible for dataset {title}")



In [11]:
DCAT,dcatGraph=create_catalog_graph(
    catalog_uri= "",
    title =  "Colorado Information Marketplace",
    description = "Driving transparency, innovation, and accountability by empowering the public with curated, accessible, discoverable, and reusable open data.",
    homepage = "http://data.colorado.gov",
    publisher_uri = "http://data.colorado.gov",
    publisher_name = "Business Intelligence Center")

#DCAT,dcatGraph=createDcat()

for title,dct in sorted(cimDatasets.items()): 
  #  print(title) 
    url=dct['permalink']
    title=dct['resource']['name']
    w4x4=dct['resource']['id']
    apiLink = f"https://data.colorado.gov/resource/{w4x4}.csv"

    description=dct['resource']['description']
    addDcatDataset(DCAT,dcatGraph,title, description, url,apiLink)



In [12]:
turtle_output = dcatGraph.serialize(format="turtle")


In [13]:
with open("cim.ttl", "w", encoding="utf-8") as f:
    f.write(turtle_output)
    f.close()

In [ ]:
# Parse the Turtle string
g = Graph()
g.parse(data=turtle_output, format="turtle")

# Define DCAT and DCTERMS namespaces
DCAT = Namespace("http://www.w3.org/ns/dcat#")
DCT = Namespace("http://purl.org/dc/terms/")

# Check for at least one dcat:Dataset
datasets = list(g.subjects(RDF.type, DCAT.Dataset))
print(f"Found {len(datasets)} dcat:Dataset(s)")

# Check for at least one dcat:Distribution
distributions = list(g.subjects(RDF.type, DCAT.Distribution))
print(f"Found {len(distributions)} dcat:Distribution(s)")

# Check for dct:language property
has_language = any(g.objects(subject=None, predicate=DCT.language))
print(f"Contains dct:language: {has_language}")

# Optionally, print a summary of the first dataset
if datasets:
    for p, o in g.predicate_objects(datasets[0]):
        print(f"{p} -> {o}")

In [12]:
def addDcatDataset_old(DCAT,dcat_graph, title, description,url): 
    # Create a sample dataset
    dataset_uri = URIRef(url)

    # Add basic dataset information
    dcat_graph.add((dataset_uri, RDF.type, DCAT.Dataset))
    dcat_graph.add((dataset_uri, DCTERMS.title, Literal(title, lang="en")))
    dcat_graph.add((dataset_uri, DCTERMS.description, Literal(description, lang="en")))

    # Add language information using the correct EU authority URIs
    english_lang_uri = URIRef("http://publications.europa.eu/resource/authority/language/ENG")

    # Add language properties
    dcat_graph.add((dataset_uri, DCTERMS.language, english_lang_uri))

    # Create distributions in different languages
    en_distribution = URIRef("https://example.org/dataset/my-multilingual-dataset/distribution/en")
    dcat_graph.add((dataset_uri, DCAT.distribution, en_distribution))


    # English distribution
    dcat_graph.add((en_distribution, RDF.type, DCAT.Distribution))
    dcat_graph.add((en_distribution, DCTERMS.title, Literal("English Version", lang="en")))
    dcat_graph.add((en_distribution, DCTERMS.language, english_lang_uri))
    dcat_graph.add((en_distribution, DCAT.mediaType, Literal("text/csv")))


    print("✅ Created sample DCAT 3 dataset with proper language metadata")

    # Serialize and display the RDF
    print("\n=== GENERATED DCAT 3 RDF (Turtle format) ===")
    dcat_graph.serialize(format="turtle")

In [60]:
for indx,row in track.iterrows():
    if 'Dataset Title' in row: 
        title = row['Dataset Title']
        description = row['Short Description']
        url = row['Permalink']
        createDcat(title, description, url)
    else:
        print("Missing title or description for row:")
        print(row)
    if indx > 5:
        break

KeyError: 'Permalink'

In [30]:
dct

{'resource': {'name': 'Zoning in Boulder County 2019',
  'id': 'yvwz-kj9z',
  'resource_name': None,
  'parent_fxf': [],
  'description': 'GIS data and locations of zoning in Boulder County in 2019.',
  'attribution': 'Boulder County',
  'attribution_link': 'https://bouldercounty.gov/',
  'contact_email': None,
  'type': 'dataset',
  'updatedAt': '2025-08-22T11:05:24.000Z',
  'createdAt': '2021-04-06T20:57:59.000Z',
  'metadata_updated_at': '2025-08-22T11:05:24.000Z',
  'data_updated_at': '2021-04-06T21:07:28.000Z',
  'page_views': {'page_views_last_week': 1,
   'page_views_last_month': 2,
   'page_views_total': 596,
   'page_views_last_week_log': 1.0,
   'page_views_last_month_log': 1.5849625007211563,
   'page_views_total_log': 9.221587121264806},
  'columns_name': ['the_geom', 'ZONING', 'SHAPE_AREA', 'SHAPE_LEN'],
  'columns_field_name': ['the_geom', 'zoning', 'shape_area', 'shape_len'],
  'columns_datatype': ['MultiPolygon', 'Text', 'Number', 'Number'],
  'columns_description': [''

In [31]:
dct['resource']

{'name': 'Zoning in Boulder County 2019',
 'id': 'yvwz-kj9z',
 'resource_name': None,
 'parent_fxf': [],
 'description': 'GIS data and locations of zoning in Boulder County in 2019.',
 'attribution': 'Boulder County',
 'attribution_link': 'https://bouldercounty.gov/',
 'contact_email': None,
 'type': 'dataset',
 'updatedAt': '2025-08-22T11:05:24.000Z',
 'createdAt': '2021-04-06T20:57:59.000Z',
 'metadata_updated_at': '2025-08-22T11:05:24.000Z',
 'data_updated_at': '2021-04-06T21:07:28.000Z',
 'page_views': {'page_views_last_week': 1,
  'page_views_last_month': 2,
  'page_views_total': 596,
  'page_views_last_week_log': 1.0,
  'page_views_last_month_log': 1.5849625007211563,
  'page_views_total_log': 9.221587121264806},
 'columns_name': ['the_geom', 'ZONING', 'SHAPE_AREA', 'SHAPE_LEN'],
 'columns_field_name': ['the_geom', 'zoning', 'shape_area', 'shape_len'],
 'columns_datatype': ['MultiPolygon', 'Text', 'Number', 'Number'],
 'columns_description': ['', '', '', ''],
 'columns_format': [

In [57]:
dct['classification']


{'categories': [],
 'tags': [],
 'domain_category': 'Zoning',
 'domain_tags': [],
 'domain_metadata': [{'key': 'Contributing-Agency-Information_Data-Source',
   'value': ''},
  {'key': 'Contributing-Agency-Information_Agency-Data-Series-Page',
   'value': ''},
  {'key': 'Contributing-Agency-Information_Citation',
   'value': 'Boulder County'},
  {'key': 'Contributing-Agency-Information_Agency-Program-Page',
   'value': 'https://bouldercounty.gov/'},
  {'key': 'Point-of-Contact_Contact-Email', 'value': 'bic-help@xentity.com'},
  {'key': 'Point-of-Contact_Contact-Name',
   'value': 'Business Intelligence Center of CO'},
  {'key': 'Dataset-Coverage_Geographic-Coverage', 'value': ''},
  {'key': 'Dataset-Coverage_Unit-of-Analysis', 'value': ''},
  {'key': 'Geospatial_Horizontal-Coordinate-System', 'value': ''},
  {'key': 'Geospatial_Coordinate-System-Disclaimer', 'value': ''},
  {'key': 'Geospatial_Horizontal-Accuracy', 'value': ''},
  {'key': 'Geospatial_Web-Display-Coordinate-System', 'va

In [32]:
for key,val in dct.items():
    print(key,val)


resource {'name': 'Zoning in Boulder County 2019', 'id': 'yvwz-kj9z', 'resource_name': None, 'parent_fxf': [], 'description': 'GIS data and locations of zoning in Boulder County in 2019.', 'attribution': 'Boulder County', 'attribution_link': 'https://bouldercounty.gov/', 'contact_email': None, 'type': 'dataset', 'updatedAt': '2025-08-22T11:05:24.000Z', 'createdAt': '2021-04-06T20:57:59.000Z', 'metadata_updated_at': '2025-08-22T11:05:24.000Z', 'data_updated_at': '2021-04-06T21:07:28.000Z', 'page_views': {'page_views_last_week': 1, 'page_views_last_month': 2, 'page_views_total': 596, 'page_views_last_week_log': 1.0, 'page_views_last_month_log': 1.5849625007211563, 'page_views_total_log': 9.221587121264806}, 'columns_name': ['the_geom', 'ZONING', 'SHAPE_AREA', 'SHAPE_LEN'], 'columns_field_name': ['the_geom', 'zoning', 'shape_area', 'shape_len'], 'columns_datatype': ['MultiPolygon', 'Text', 'Number', 'Number'], 'columns_description': ['', '', '', ''], 'columns_format': [{}, {}, {}, {}], 'd

In [58]:
dct['permalink']

'https://data.colorado.gov/d/yvwz-kj9z'

In [54]:
# Extract 'classification' and 'publisher' from each dict (if present)
results = [
    {'publisher': d.get('value')}
    for d in dct['classification']['domain_metadata']
    if 'Publisher_Publisher-Name' in d['key']
]

In [55]:
results

[{'publisher': 'Business Intelligence Center of CO'}]

In [51]:
# See what keys are present in each dict
for d in dct['classification']['domain_metadata']:
    
    print(d.keys())
    print(d.get('key'),d.get('value'))

dict_keys(['key', 'value'])
Contributing-Agency-Information_Data-Source 
dict_keys(['key', 'value'])
Contributing-Agency-Information_Agency-Data-Series-Page 
dict_keys(['key', 'value'])
Contributing-Agency-Information_Citation Boulder County
dict_keys(['key', 'value'])
Contributing-Agency-Information_Agency-Program-Page https://bouldercounty.gov/
dict_keys(['key', 'value'])
Point-of-Contact_Contact-Email bic-help@xentity.com
dict_keys(['key', 'value'])
Point-of-Contact_Contact-Name Business Intelligence Center of CO
dict_keys(['key', 'value'])
Dataset-Coverage_Geographic-Coverage 
dict_keys(['key', 'value'])
Dataset-Coverage_Unit-of-Analysis 
dict_keys(['key', 'value'])
Geospatial_Horizontal-Coordinate-System 
dict_keys(['key', 'value'])
Geospatial_Coordinate-System-Disclaimer 
dict_keys(['key', 'value'])
Geospatial_Horizontal-Accuracy 
dict_keys(['key', 'value'])
Geospatial_Web-Display-Coordinate-System 
dict_keys(['key', 'value'])
Geospatial_Collection-Method 
dict_keys(['key', 'valu

In [33]:
dct['classification']

{'categories': [],
 'tags': [],
 'domain_category': 'Zoning',
 'domain_tags': [],
 'domain_metadata': [{'key': 'Contributing-Agency-Information_Data-Source',
   'value': ''},
  {'key': 'Contributing-Agency-Information_Agency-Data-Series-Page',
   'value': ''},
  {'key': 'Contributing-Agency-Information_Citation',
   'value': 'Boulder County'},
  {'key': 'Contributing-Agency-Information_Agency-Program-Page',
   'value': 'https://bouldercounty.gov/'},
  {'key': 'Point-of-Contact_Contact-Email', 'value': 'bic-help@xentity.com'},
  {'key': 'Point-of-Contact_Contact-Name',
   'value': 'Business Intelligence Center of CO'},
  {'key': 'Dataset-Coverage_Geographic-Coverage', 'value': ''},
  {'key': 'Dataset-Coverage_Unit-of-Analysis', 'value': ''},
  {'key': 'Geospatial_Horizontal-Coordinate-System', 'value': ''},
  {'key': 'Geospatial_Coordinate-System-Disclaimer', 'value': ''},
  {'key': 'Geospatial_Horizontal-Accuracy', 'value': ''},
  {'key': 'Geospatial_Web-Display-Coordinate-System', 'va

In [ ]:
dct['classification']['publisher']